# CogniSync → TMLR: Master Kaggle Experiment Suite

This master notebook runs the complete experimental testbed for the **TMLR (Transactions on Machine Learning Research)** submission.

### ⚡ Kaggle Session Settings
1. **Accelerator:** `GPU T4 x1` (Settings panel on right)
2. **Internet:** `ON` (Settings panel on right)
3. **Persistence:** `Filesystem / Variables ON`

### ⏱ Execution Structure (Tuned for 12-Hour Session Limits)
The total suite takes ~25–30 hours on a single T4 GPU. It is partitioned into **three 12-hour session blocks** (plus a 10-minute smoke test):
- **Block 0 (~10 min):** Quick Smoke Test (Validates all 4 tracks on small subsets)
- **Block 1 (~8–10 h):** **Track A — Full Retrieval Audit** (BEIR 7-dataset sweep, contamination, alpha decomposition, CE budget)
- **Block 2 (~8–10 h):** **Track B — Security & Compliance** (6-level attack ladder, LLM compliance ACR, detector transfer, cost curve)
- **Block 3 (~8–9 h):** **Track C & D — Memory & Latency** (LongMemEval episodic graph, timestamp-shuffling ablation, latency profiling)

All runs **checkpoint automatically** to disk; if a session ends mid-run, re-running the cell will resume where it stopped.


## 1. Setup & Environment
Installs all dependencies, extracts the self-contained codebase, and sets up persistent cache & output directories.


In [ ]:
# 1. Install required packages
!pip install -q sentence-transformers faiss-cpu bm25s PyStemmer rank_bm25 beir datasets transformers accelerate

# Optional: 4-bit quantization for Qwen-7B (uncomment if needed):
# !pip install -q bitsandbytes


In [ ]:
import os, sys, shutil, subprocess, time, io, zipfile, base64
from pathlib import Path
import pandas as pd

# 1. Set persistent directory environment variables
os.environ['COGNISYNC_ROOT']  = '/kaggle/working/cognisync_out'
os.environ['COGNISYNC_CACHE'] = '/kaggle/working/cognisync_cache'
os.environ['COGNISYNC_DATA']  = '/kaggle/working/cognisync_data'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

for d in ['/kaggle/working/cognisync_out/results', '/kaggle/working/cognisync_out/logs', '/kaggle/working/cognisync_out/figures', '/kaggle/working/cognisync_cache', '/kaggle/working/cognisync_data']:
    Path(d).mkdir(parents=True, exist_ok=True)

# 2. Auto-extract self-contained experiment code bundle
TARGET_DIR = Path('/kaggle/working/cognisync-tmlr')
EMBEDDED_BUNDLE = '''...'''

if not (TARGET_DIR / 'exp_a' / 'a1_beir_full_corpus.py').exists():
    print('>>> Extracting self-contained experiment suite to:', TARGET_DIR)
    TARGET_DIR.mkdir(parents=True, exist_ok=True)
    zip_data = base64.b64decode(EMBEDDED_BUNDLE.strip())
    with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
        zf.extractall(TARGET_DIR)
    print('>>> Extraction complete!')
else:
    print('>>> Experiment suite already available at:', TARGET_DIR)

SUITE_DIR = TARGET_DIR
if str(SUITE_DIR) not in sys.path:
    sys.path.insert(0, str(SUITE_DIR))
os.chdir(str(SUITE_DIR))

import torch
print(f'>>> PyTorch {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'>>> GPU Device: {torch.cuda.get_device_name(0)}')
else:
    print('>>> WARNING: Running on CPU. Switch accelerator to GPU T4 x1 for full production runs.')

# Live streaming execution helper
def run_cmd(cmd: str):
    print(f'\n' + '='*70)
    print(f'>>> EXEC: {cmd}')
    print('='*70)
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.stdout.close()
    ret = proc.wait()
    elapsed = time.time() - t0
    print(f'\n>>> DONE in {elapsed/60:.2f} min (Exit Code: {ret})')
    return ret


---
## Mode 0: Fast Smoke Test (~10–15 mins)
Runs lightweight verification across all 4 experimental tracks on small subsets to confirm all modules, models, and metric aggregations run cleanly.


In [ ]:
# 1. Track A: Quick BEIR + Contamination + Alpha decomposition on tiny subset
run_cmd('python -m exp_a.a1_beir_full_corpus --datasets scifact nfcorpus --rerankers none ms-marco')
run_cmd('python -m exp_a.a2_contamination --contrast train-vs-dev --n-queries 30')
run_cmd('python -m exp_a.a3_alpha_decomposition --datasets scifact --query-limit 30')

# 2. Track B: Attack ladder & detector comparison
run_cmd('python -m exp_b.b1_b2_attack_ladder --dataset scifact --n-queries 30 --levels 0 1 2')
run_cmd('python -m exp_b.b4_detector_comparison --skip-transformers')

# 3. Track C: Episodic graph on LongMemEval
run_cmd('python -m exp_c.c1_longmemeval --variant s --n-instances 10')

# 4. Track D: Latency profile (CUDA / CPU)
run_cmd('python -m exp_d.d1_latency_profile --sizes 10000 --device cuda --n-queries 20')

print('\n' + '='*70)
print('>>> ALL SMOKE TESTS COMPLETED SUCCESSFULLY!')
print('='*70)


---
## 12-Hour Session Block 1: Track A — Full Retrieval Audit (~8–10 hours)
Addresses CIKM Table 1, Table 4, and Table 5:
1. **A1 (`a1_beir_full_corpus.py`):** Full-corpus zero-shot BEIR across 7 datasets with complete factorial over first-stage × reranker (builds persistent embedding cache).
2. **A2 (`a2_contamination.py`):** Quantifies train-vs-dev contamination on MS MARCO, reranker provenance (`bge-reranker` vs `ms-marco`), and mixture membership.
3. **A3 (`a3_alpha_decomposition.py`):** Decomposes learned alpha (Random Forest vs undisclosed fallback rule vs best-fixed-alpha vs oracle-alpha).
4. **A4 (`a4_ce_budget.py`):** Sweeps cross-encoder rerank depth budget (5 to 200).


In [ ]:
# A1: Full BEIR Sweep (builds persistent embedding cache)
run_cmd('python -m exp_a.a1_beir_full_corpus --all --rerankers none ms-marco')

# A2: Contamination & Reranker Provenance
run_cmd('python -m exp_a.a2_contamination --contrast train-vs-dev')
run_cmd('python -m exp_a.a2_contamination --contrast reranker-provenance')
run_cmd('python -m exp_a.a2_contamination --contrast mixture-membership --a1-results /kaggle/working/cognisync_out/results/a1_beir_full_corpus.csv')

# A3: Alpha Decomposition
run_cmd('python -m exp_a.a3_alpha_decomposition --datasets scifact nfcorpus fiqa --with-reranker')

# A4: Cross-Encoder Depth Budget Sweep
run_cmd('python -m exp_a.a4_ce_budget --datasets scifact nfcorpus fiqa --depths 5 10 20 50 100 200')


In [ ]:
# Display Track A Results Tables
R = Path('/kaggle/working/cognisync_out/results')
track_a_tables = ['a2_contamination_gap.csv', 'a3_alpha_diagnostics.csv', 'a3_alpha_decomposition.csv', 'a1_beir_full_corpus__pivot.csv', 'a4_ce_budget.csv']
for tbl in track_a_tables:
    p = R / tbl
    if p.exists():
        print(f'\n=== {tbl} ===')
        df = pd.read_csv(p)
        display(df.round(4))
    else:
        print(f'[!] {tbl} not yet generated')


---
## 12-Hour Session Block 2: Track B — Security & Downstream Compliance (~8–10 hours)
Addresses CIKM Table 7, Table 9, and reviewer feedback on realistic adversarial threat models:
1. **B1/B2 (`b1_b2_attack_ladder.py`):** 6-level attack ladder (L0 naive to L5 adaptive) evaluated with oracle vs deployable held-out centroid.
2. **B4 (`b4_detector_comparison.py`):** 3-feature filter vs DeBERTa-v3 and Prompt-Guard across transfer splits.
3. **B3 (`b3_downstream_compliance.py`):** Downstream LLM instruction compliance (ACR) on Qwen2.5-3B + tool spoofing.
4. **B5 (`b5_defense_cost_curve.py`):** Corrected defense cost curve at realistic base poisoning rates ($10^{-2}$ to $10^{-5}$).


In [ ]:
# B1/B2: Attack Ladder (Oracle vs Held-out Centroid)
run_cmd('python -m exp_b.b1_b2_attack_ladder --dataset scifact --n-queries 200 --centroid-sources oracle holdout --levels 0 1 2 3 4 5')
run_cmd('python -m exp_b.b1_b2_attack_ladder --dataset fiqa --n-queries 200 --centroid-sources holdout --levels 0 1 2 3 4 5')

# B4: Detector Comparison Across Injection Datasets
run_cmd('python -m exp_b.b4_detector_comparison --corpora deepset jailbreak')

# B3: Downstream LLM Compliance & Tool Spoofing (Loads LLM once)
run_cmd('python -m exp_b.b3_downstream_compliance --dataset scifact --n-queries 100 --models qwen2.5-3b --levels 0 2 3 4')
run_cmd('python -m exp_b.b3_downstream_compliance --tool-spoofing --models qwen2.5-3b --n-queries 50')

# B5: Defense Cost Curve across Poisoning Base Rates
run_cmd('python -m exp_b.b5_defense_cost_curve --dataset scifact --n-queries 150')


In [ ]:
# Display Track B Results Tables
track_b_tables = ['b1_b2_attack_ladder.csv', 'b4_detector_comparison.csv', 'b3_downstream_compliance.csv', 'b5_defense_cost_curve.csv']
for tbl in track_b_tables:
    p = R / tbl
    if p.exists():
        print(f'\n=== {tbl} ===')
        df = pd.read_csv(p)
        display(df.round(4))
    else:
        print(f'[!] {tbl} not yet generated')


---
## 12-Hour Session Block 3: Track C & D — Episodic Memory & Latency (~8–9 hours)
Addresses CIKM Table 10 and Section 4.2 (Episodic Memory implementation + cost profiling):
1. **C1 (`c1_longmemeval.py`):** Benchmark episodic graph on LongMemEval (ICLR 2025) on `temporal-reasoning` and `knowledge-update` subsets (+ LLM QA accuracy).
2. **C2 (`c2_graph_ablation.py`):** Edge type ablations + timestamp shuffling negative control.
3. **D1 (`d1_latency_profile.py`):** Latency breakdown across pipeline stages on real text (GPU & CPU at 10k/100k scale).


In [ ]:
# C1: LongMemEval Evaluation (Retrieval + QA with Qwen2.5-3B)
run_cmd('python -m exp_c.c1_longmemeval --variant s --n-instances 100')
run_cmd('python -m exp_c.c1_longmemeval --variant s --n-instances 100 --with-qa --qa-model qwen2.5-3b')

# C2: Graph Ablation & Timestamp Shuffling Negative Control
run_cmd('python -m exp_c.c2_graph_ablation --n-instances 100')

# D1: Latency Profile on Real Text (GPU & CPU)
run_cmd('python -m exp_d.d1_latency_profile --sizes 10000 100000 --device cuda')
run_cmd('python -m exp_d.d1_latency_profile --sizes 10000 --device cpu')


In [ ]:
# Display Track C & D Results Tables
track_cd_tables = ['c1_longmemeval.csv', 'c2_graph_ablation.csv', 'd1_latency_profile.csv']
for tbl in track_cd_tables:
    p = R / tbl
    if p.exists():
        print(f'\n=== {tbl} ===')
        df = pd.read_csv(p)
        display(df.round(4))
    else:
        print(f'[!] {tbl} not yet generated')


---
## 4. Archive & Download All Results
Packages all results (`results/*.csv`), per-query significance metrics (`results/*__per_query.csv`), and fully resolved configs (`logs/*.json`) into a single downloadable zip file.


In [ ]:
import shutil
from IPython.display import FileLink

out_dir = '/kaggle/working/cognisync_out'
zip_path = '/kaggle/working/cognisync_results'

if Path(out_dir).exists():
    shutil.make_archive(zip_path, 'zip', out_dir)
    zip_full = zip_path + '.zip'
    mb = os.path.getsize(zip_full) / 1024 / 1024
    print(f'>>> Successfully created: {zip_full} ({mb:.2f} MB)')
    display(FileLink('cognisync_results.zip'))
else:
    print('>>> No results found in', out_dir)
